# 🗄️ SQL Training — Sessions 5 to 9
**Trainer: Raamashaamy**

This notebook covers:
- **Session 5** — SQL Basics (DDL, DQL, DML)
- **Session 6** — SQL Aggregations (GROUP BY, HAVING)
- **Session 7** — Joins & Unions
- **Session 8** — Intermediate SQL (Subqueries, CASE, COALESCE, CTE)
- **Session 9** — Analytical SQL (Window Functions, Pivot)

---
### 🛠️ How this notebook works

We use **SQLite** — a lightweight database that runs entirely in Python, no server needed.
All demo tables are created and loaded in the **Setup** section below.
Every SQL session then queries those same tables.

> 💡 Run **Setup first**, then jump to any session you need.

---
## ⚙️ Setup — Create Demo Database & Load Data

We model a small company with five tables:

| Table | Description |
|-------|-------------|
| `employees` | Employee master data |
| `departments` | Department names and budgets |
| `orders` | Customer orders |
| `products` | Product catalog |
| `order_items` | Line items linking orders to products |

Run this cell **once** before any other session.

In [0]:
import sqlite3
import pandas as pd

# ── Helper: run a SQL query and return a DataFrame ──────────────────────────
con = sqlite3.connect(":memory:")   # in-memory database — no file needed

def sql(query):
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(query, con)

def run(ddl_or_dml):
    """Execute DDL / DML statements (CREATE, INSERT, UPDATE, DELETE)."""
    con.executescript(ddl_or_dml)
    con.commit()

# ── Create Tables ────────────────────────────────────────────────────────────
run("""
CREATE TABLE IF NOT EXISTS departments (
    dept_id     INTEGER PRIMARY KEY,
    dept_name   TEXT    NOT NULL,
    location    TEXT,
    budget      REAL
);

CREATE TABLE IF NOT EXISTS employees (
    emp_id      TEXT    PRIMARY KEY,
    emp_name    TEXT    NOT NULL,
    dept_id     INTEGER REFERENCES departments(dept_id),
    salary      REAL,
    hire_date   TEXT,
    manager_id  TEXT
);

CREATE TABLE IF NOT EXISTS products (
    product_id  TEXT    PRIMARY KEY,
    product_name TEXT   NOT NULL,
    category    TEXT,
    unit_price  REAL
);

CREATE TABLE IF NOT EXISTS orders (
    order_id    TEXT    PRIMARY KEY,
    customer    TEXT,
    region      TEXT,
    order_date  TEXT,
    status      TEXT
);

CREATE TABLE IF NOT EXISTS order_items (
    item_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id    TEXT    REFERENCES orders(order_id),
    product_id  TEXT    REFERENCES products(product_id),
    quantity    INTEGER,
    discount    REAL    DEFAULT 0
);
""")

# ── Insert Demo Data ──────────────────────────────────────────────────────────
run("""
INSERT OR IGNORE INTO departments VALUES
  (1, 'Engineering', 'Bangalore',  5000000),
  (2, 'Finance',     'Chennai',    2000000),
  (3, 'Sales',       'Mumbai',     3000000),
  (4, 'HR',          'Chennai',    1500000),
  (5, 'Marketing',   'Delhi',      2500000);

INSERT OR IGNORE INTO employees VALUES
  ('E001', 'Arjun Sharma',    1, 95000,  '2019-06-15', NULL),
  ('E002', 'Priya Nair',      2, 72000,  '2020-03-01', 'E001'),
  ('E003', 'Karthik Rajan',   1, 87000,  '2018-11-20', 'E001'),
  ('E004', 'Deepa Menon',     4, 58000,  '2021-07-10', 'E001'),
  ('E005', 'Ravi Kumar',      3, 66000,  '2020-09-05', 'E001'),
  ('E006', 'Anjali Verma',    3, 74000,  '2019-01-22', 'E005'),
  ('E007', 'Suresh Pillai',   1, 110000, '2017-04-03', NULL),
  ('E008', 'Meena Iyer',      2, 68000,  '2022-02-14', 'E002'),
  ('E009', 'Vikram Patel',    5, 82000,  '2021-11-30', 'E001'),
  ('E010', 'Lakshmi Das',     NULL, 54000,'2023-05-01', 'E004');

INSERT OR IGNORE INTO products VALUES
  ('P01', 'Laptop Pro 15',    'Electronics',   85000),
  ('P02', 'Wireless Mouse',   'Accessories',    1200),
  ('P03', 'Mechanical Keyboard','Accessories',  4500),
  ('P04', 'Monitor 27 inch',  'Electronics',   22000),
  ('P05', 'USB-C Hub',        'Accessories',    2800),
  ('P06', 'Noise Cancel Headphones','Audio',   18000),
  ('P07', 'Webcam HD',        'Electronics',    5500),
  ('P08', 'Standing Desk',    'Furniture',     35000);

INSERT OR IGNORE INTO orders VALUES
  ('ORD001', 'TechCorp',    'South', '2024-01-10', 'Delivered'),
  ('ORD002', 'InfoSys Ltd', 'South', '2024-01-15', 'Delivered'),
  ('ORD003', 'Wipro',       'South', '2024-02-01', 'Pending'),
  ('ORD004', 'HCL',         'North', '2024-02-20', 'Delivered'),
  ('ORD005', 'Infosys Ltd', 'South', '2024-03-05', 'Cancelled'),
  ('ORD006', 'TechCorp',    'South', '2024-03-18', 'Delivered'),
  ('ORD007', 'MindTree',    'East',  '2024-04-02', 'Pending'),
  ('ORD008', 'HCL',         'North', '2024-04-25', 'Delivered'),
  ('ORD009', 'Zensar',      'West',  '2024-05-10', 'Delivered'),
  ('ORD010', 'TechCorp',    'South', '2024-05-22', 'Pending');

INSERT OR IGNORE INTO order_items (order_id, product_id, quantity, discount) VALUES
  ('ORD001','P01', 2, 0.05), ('ORD001','P02', 5, 0.00),
  ('ORD002','P03', 3, 0.10), ('ORD002','P06', 1, 0.00),
  ('ORD003','P04', 4, 0.05), ('ORD003','P05', 6, 0.00),
  ('ORD004','P01', 1, 0.00), ('ORD004','P07', 2, 0.05),
  ('ORD005','P08', 1, 0.15),
  ('ORD006','P02',10, 0.00), ('ORD006','P03', 5, 0.05),
  ('ORD007','P01', 3, 0.10), ('ORD007','P06', 2, 0.00),
  ('ORD008','P04', 2, 0.00), ('ORD008','P05', 4, 0.05),
  ('ORD009','P07', 6, 0.00), ('ORD009','P08', 1, 0.10),
  ('ORD010','P01', 1, 0.05), ('ORD010','P02', 8, 0.00);
""")

print("✅ Database ready. Tables: departments, employees, products, orders, order_items")

---
## 📘 Session 5: SQL Basics

### SQL Command Categories

| Category | What it does | Examples |
|----------|-------------|----------|
| **DDL** — Data Definition | Define structure | `CREATE`, `ALTER`, `DROP` |
| **DML** — Data Manipulation | Change data | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** — Data Query | Read data | `SELECT` |

---
### 5.1 DDL — CREATE TABLE

The tables were already created in Setup. Let's look at the structure.

In [0]:
# See all tables in our database
sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

In [0]:
# Preview the employees table
sql("SELECT * FROM employees")

In [0]:
sql("SELECT * FROM departments")

### 5.2 DQL — SELECT (Querying Data)

The `SELECT` statement is the most-used SQL command. Clause order:

```sql
SELECT   columns
FROM     table
WHERE    filter rows
ORDER BY sort result
LIMIT    cap rows returned
```

In [0]:
# Select specific columns
sql("""
SELECT emp_id, emp_name, salary
FROM   employees
ORDER BY salary DESC
""")

In [0]:
# WHERE — filter rows
# Employees with salary above 80000 in Engineering (dept_id = 1)
sql("""
SELECT emp_name, salary
FROM   employees
WHERE  salary > 80000
  AND  dept_id = 1
ORDER BY salary DESC
""")

In [0]:
# LIKE — pattern matching (% = any characters, _ = one character)
sql("""
SELECT emp_name
FROM   employees
WHERE  emp_name LIKE 'A%'   -- names starting with A
""")

In [0]:
# IN — match against a list of values
sql("""
SELECT emp_name, dept_id
FROM   employees
WHERE  dept_id IN (1, 3, 5)   -- Engineering, Sales, Marketing
ORDER BY dept_id
""")

### 5.3 DML — INSERT, UPDATE, DELETE

In [0]:
# INSERT — add a new row
run("""
INSERT OR IGNORE INTO employees (emp_id, emp_name, dept_id, salary, hire_date)
VALUES ('E011', 'Test Employee', 3, 62000, '2024-06-01');
""")

sql("SELECT * FROM employees WHERE emp_id = 'E011'")

In [0]:
# UPDATE — modify existing rows (ALWAYS use WHERE or you update ALL rows!)
run("UPDATE employees SET salary = 65000 WHERE emp_id = 'E011';")
sql("SELECT emp_name, salary FROM employees WHERE emp_id = 'E011'")

In [0]:
# DELETE — remove rows (again, ALWAYS use WHERE)
run("DELETE FROM employees WHERE emp_id = 'E011';")
sql("SELECT COUNT(*) AS remaining_employees FROM employees")

---
## 📘 Session 6: SQL Aggregations

Aggregations **summarise** many rows into fewer rows.

| Function | What it returns |
|----------|-----------------|
| `COUNT(*)` | Number of rows |
| `COUNT(col)` | Non-NULL values in column |
| `SUM(col)` | Total |
| `AVG(col)` | Average |
| `MIN(col)` | Smallest value |
| `MAX(col)` | Largest value |

### 6.1 Basic Aggregations

In [0]:
sql("""
SELECT
    COUNT(*)          AS total_employees,
    COUNT(manager_id) AS employees_with_manager,   -- NULLs not counted
    ROUND(AVG(salary), 2) AS avg_salary,
    MIN(salary)       AS min_salary,
    MAX(salary)       AS max_salary,
    SUM(salary)       AS total_payroll
FROM employees
""")

### 6.2 GROUP BY

`GROUP BY` splits the rows into groups by one or more columns and applies the aggregate to **each group** separately.

In [0]:
# Average salary and headcount per department
sql("""
SELECT
    d.dept_name,
    COUNT(e.emp_id)           AS headcount,
    ROUND(AVG(e.salary), 0)   AS avg_salary,
    MAX(e.salary)             AS top_salary
FROM       employees   e
LEFT JOIN  departments d ON e.dept_id = d.dept_id
GROUP BY   d.dept_name
ORDER BY   avg_salary DESC
""")

### 6.3 HAVING

`HAVING` filters **after** aggregation (like a `WHERE` for group results).

👉 Rule: Use `WHERE` to filter **rows**, use `HAVING` to filter **groups**.

In [0]:
# Only show departments where average salary exceeds 70000
sql("""
SELECT
    d.dept_name,
    COUNT(e.emp_id)           AS headcount,
    ROUND(AVG(e.salary), 0)   AS avg_salary
FROM       employees   e
LEFT JOIN  departments d ON e.dept_id = d.dept_id
GROUP BY   d.dept_name
HAVING     AVG(e.salary) > 70000
ORDER BY   avg_salary DESC
""")

In [0]:
# Products ordered more than 3 times across all orders
sql("""
SELECT
    p.product_name,
    COUNT(oi.order_id)   AS times_ordered,
    SUM(oi.quantity)     AS total_units
FROM       order_items oi
JOIN       products    p  ON oi.product_id = p.product_id
GROUP BY   p.product_name
HAVING     COUNT(oi.order_id) > 2
ORDER BY   total_units DESC
""")

---
## 📘 Session 7: Joins & Unions

### Why Joins?
Data is stored in **separate tables** to avoid redundancy. Joins combine them using a matching key.

| Join Type | Returns |
|-----------|--------|
| `INNER JOIN` | Only matching rows from both tables |
| `LEFT JOIN` | All rows from left + matching from right (NULL if no match) |
| `RIGHT JOIN` | All rows from right + matching from left |
| `FULL OUTER JOIN` | All rows from both tables |

---
### 7.1 INNER JOIN

In [0]:
# Employees WITH a department (E010 has NULL dept_id — excluded by INNER JOIN)
sql("""
SELECT
    e.emp_id,
    e.emp_name,
    d.dept_name,
    d.location,
    e.salary
FROM  employees   e
INNER JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name, e.salary DESC
""")

### 7.2 LEFT JOIN

Returns **all rows from the left table**, even if there is no match in the right table. Unmatched right-side columns show `NULL`.

In [0]:
# ALL employees — even Lakshmi (no dept) will appear with NULL dept_name
sql("""
SELECT
    e.emp_id,
    e.emp_name,
    d.dept_name,
    e.salary
FROM  employees   e
LEFT JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name NULLS LAST
""")

In [0]:
# Find departments that have NO employees assigned
sql("""
SELECT d.dept_name
FROM   departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
WHERE  e.emp_id IS NULL
""")

### 7.3 Multi-Table Join

In [0]:
# Order revenue: join orders → order_items → products
sql("""
SELECT
    o.order_id,
    o.customer,
    o.order_date,
    o.status,
    p.product_name,
    oi.quantity,
    p.unit_price,
    ROUND(oi.quantity * p.unit_price * (1 - oi.discount), 2) AS line_total
FROM   orders      o
JOIN   order_items oi ON o.order_id    = oi.order_id
JOIN   products    p  ON oi.product_id = p.product_id
ORDER BY o.order_date, o.order_id
""")

### 7.4 UNION vs UNION ALL

| | What it does |
|---|---|
| `UNION` | Combines results and **removes duplicates** (slower) |
| `UNION ALL` | Combines results and **keeps all rows** (faster) |

Both queries must return the **same number of columns** with compatible types.

In [0]:
# High earners (>85000) UNION with employees hired in 2017
# Some may appear in both — UNION removes the duplicate
sql("""
SELECT emp_name, salary, 'High Earner' AS reason
FROM   employees
WHERE  salary > 85000

UNION

SELECT emp_name, salary, 'Early Joiner'
FROM   employees
WHERE  hire_date < '2018-01-01'
ORDER BY salary DESC
""")

In [0]:
# UNION ALL — keeps all rows even if repeated
sql("""
SELECT emp_name, 'High Earner' AS tag FROM employees WHERE salary > 85000
UNION ALL
SELECT emp_name, 'Early Joiner'       FROM employees WHERE hire_date < '2018-01-01'
""")

---
## 📘 Session 8: Intermediate SQL

---
### 8.1 Subqueries

A **subquery** is a query nested inside another query. It runs first and its result is used by the outer query.

In [0]:
# Employees earning above the company average
sql("""
SELECT emp_name, salary
FROM   employees
WHERE  salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC
""")

In [0]:
# Subquery in FROM (derived table / inline view)
# Get departments where avg salary > company avg
sql("""
SELECT dept_name, dept_avg
FROM (
    SELECT d.dept_name, ROUND(AVG(e.salary), 0) AS dept_avg
    FROM   employees   e
    JOIN   departments d ON e.dept_id = d.dept_id
    GROUP BY d.dept_name
) AS dept_summary
WHERE dept_avg > (SELECT AVG(salary) FROM employees)
ORDER BY dept_avg DESC
""")

### 8.2 CASE WHEN

`CASE WHEN` is SQL's if/else — used to create a new column based on conditions.

In [0]:
# Classify employees by salary band
sql("""
SELECT
    emp_name,
    salary,
    CASE
        WHEN salary >= 90000 THEN 'Senior'
        WHEN salary >= 70000 THEN 'Mid'
        ELSE                      'Junior'
    END AS salary_band
FROM  employees
ORDER BY salary DESC
""")

In [0]:
# CASE inside aggregate — count employees per band
sql("""
SELECT
    COUNT(CASE WHEN salary >= 90000 THEN 1 END) AS senior_count,
    COUNT(CASE WHEN salary BETWEEN 70000 AND 89999 THEN 1 END) AS mid_count,
    COUNT(CASE WHEN salary < 70000  THEN 1 END) AS junior_count
FROM employees
""")

### 8.3 NULL Handling — IFNULL & COALESCE

- `IFNULL(col, default)` — return default if col is NULL *(SQLite / MySQL)*
- `COALESCE(col1, col2, ...)` — return the **first non-NULL** value from the list *(ANSI standard — works everywhere)*

In [0]:
sql("""
SELECT
    emp_name,
    dept_id,
    manager_id,
    IFNULL(CAST(dept_id AS TEXT),   'Unassigned') AS dept_label,
    COALESCE(manager_id, emp_id)                  AS reports_to    -- self if no manager
FROM employees
ORDER BY dept_id NULLS LAST
""")

### 8.4 CTE — WITH Clause (Common Table Expression)

A CTE gives a **name** to a subquery so you can reference it cleanly — like a temporary view that lives only for that query.

Benefits over subqueries:
- Readable top-to-bottom (like steps)
- Can be referenced multiple times
- Easier to debug (test each CTE separately)

In [0]:
# CTE: calculate order revenue, then find top customers
sql("""
WITH order_revenue AS (
    SELECT
        o.order_id,
        o.customer,
        o.status,
        ROUND(SUM(oi.quantity * p.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM   orders      o
    JOIN   order_items oi ON o.order_id    = oi.order_id
    JOIN   products    p  ON oi.product_id = p.product_id
    GROUP BY o.order_id, o.customer, o.status
),
customer_total AS (
    SELECT customer, SUM(revenue) AS total_revenue
    FROM   order_revenue
    WHERE  status = 'Delivered'
    GROUP BY customer
)
SELECT *
FROM   customer_total
ORDER BY total_revenue DESC
""")

---
## 📘 Session 9: Analytical SQL

### Window Functions

Window functions perform calculations **across a set of rows related to the current row** — without collapsing them into a single result like GROUP BY.

Syntax:
```sql
function_name()  OVER (
    PARTITION BY  col1         -- define the group (window)
    ORDER BY      col2         -- define the order within the window
    ROWS BETWEEN  ...          -- optional frame
)
```

| Function | Purpose |
|----------|---------|
| `ROW_NUMBER()` | Unique sequential row number |
| `RANK()` | Rank with gaps on ties |
| `DENSE_RANK()` | Rank without gaps on ties |
| `LAG()` | Value from the previous row |
| `LEAD()` | Value from the next row |
| `SUM() OVER` | Running / cumulative total |
| `AVG() OVER` | Running average |

---
### 9.1 ROW_NUMBER, RANK, DENSE_RANK

In [0]:
# Rank employees by salary within each department
sql("""
SELECT
    e.emp_name,
    d.dept_name,
    e.salary,
    ROW_NUMBER() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS row_num,
    RANK()       OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS rank,
    DENSE_RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS dense_rank
FROM  employees   e
JOIN  departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name, e.salary DESC
""")

In [0]:
# Top earner per department — wrap ranking in a CTE, then filter
sql("""
WITH ranked AS (
    SELECT
        e.emp_name,
        d.dept_name,
        e.salary,
        RANK() OVER (PARTITION BY e.dept_id ORDER BY e.salary DESC) AS rnk
    FROM  employees   e
    JOIN  departments d ON e.dept_id = d.dept_id
)
SELECT emp_name, dept_name, salary
FROM   ranked
WHERE  rnk = 1
ORDER BY salary DESC
""")

### 9.2 Running Total (Cumulative SUM)

In [0]:
# Cumulative revenue per order (sorted by date)
sql("""
WITH daily_rev AS (
    SELECT
        o.order_date,
        o.order_id,
        ROUND(SUM(oi.quantity * p.unit_price * (1 - oi.discount)), 2) AS order_revenue
    FROM   orders      o
    JOIN   order_items oi ON o.order_id    = oi.order_id
    JOIN   products    p  ON oi.product_id = p.product_id
    GROUP BY o.order_date, o.order_id
)
SELECT
    order_date,
    order_id,
    order_revenue,
    ROUND(SUM(order_revenue) OVER (ORDER BY order_date, order_id
                                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2)
        AS cumulative_revenue
FROM daily_rev
ORDER BY order_date, order_id
""")

### 9.3 LAG & LEAD — Compare with Previous / Next Row

In [0]:
# Month-over-month salary comparison (simulated using hire order)
sql("""
SELECT
    emp_name,
    hire_date,
    salary,
    LAG(salary)  OVER (ORDER BY hire_date) AS prev_emp_salary,
    LEAD(salary) OVER (ORDER BY hire_date) AS next_emp_salary,
    salary - LAG(salary) OVER (ORDER BY hire_date) AS salary_diff
FROM employees
ORDER BY hire_date
""")

### 9.4 Pivot Table (CASE-based)

SQL doesn't have a native `PIVOT` keyword in most databases (except SQL Server and BigQuery). The standard approach uses `CASE WHEN` inside aggregations.

In [0]:
# Headcount per department per salary band — pivot-style output
sql("""
SELECT
    d.dept_name,
    COUNT(CASE WHEN e.salary >= 90000                     THEN 1 END) AS senior,
    COUNT(CASE WHEN e.salary BETWEEN 70000 AND 89999      THEN 1 END) AS mid,
    COUNT(CASE WHEN e.salary < 70000                      THEN 1 END) AS junior,
    COUNT(e.emp_id)                                                    AS total
FROM  employees   e
JOIN  departments d ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY d.dept_name
""")

In [0]:
# Orders by status across regions — classic pivot
sql("""
SELECT
    region,
    COUNT(CASE WHEN status = 'Delivered'  THEN 1 END) AS delivered,
    COUNT(CASE WHEN status = 'Pending'    THEN 1 END) AS pending,
    COUNT(CASE WHEN status = 'Cancelled'  THEN 1 END) AS cancelled,
    COUNT(*) AS total_orders
FROM  orders
GROUP BY region
ORDER BY region
""")

### 9.5 SUM OVER PARTITION — % Contribution

In [0]:
# Each employee's salary as % of their department's total payroll
sql("""
SELECT
    e.emp_name,
    d.dept_name,
    e.salary,
    SUM(e.salary) OVER (PARTITION BY e.dept_id)   AS dept_payroll,
    ROUND(100.0 * e.salary /
          SUM(e.salary) OVER (PARTITION BY e.dept_id), 1) AS pct_of_dept
FROM  employees   e
JOIN  departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name, e.salary DESC
""")

---
## ✅ Quick Recap — Sessions 5 to 9

| Session | Key Concept | Remember |
|---------|-------------|----------|
| 5 — Basics | DDL / DML / DQL | Always use WHERE with UPDATE/DELETE |
| 6 — Aggregations | GROUP BY + HAVING | WHERE filters rows, HAVING filters groups |
| 7 — Joins | INNER / LEFT / UNION | LEFT JOIN keeps all left rows |
| 8 — Intermediate | Subqueries, CASE, COALESCE, CTE | CTEs make complex queries readable |
| 9 — Analytical | Window functions, Pivot | PARTITION BY = define the window group |

---
### 🧩 Demo Tables Reference

| Table | Key Columns |
|-------|-------------|
| `employees` | emp_id, emp_name, dept_id, salary, hire_date, manager_id |
| `departments` | dept_id, dept_name, location, budget |
| `orders` | order_id, customer, region, order_date, status |
| `products` | product_id, product_name, category, unit_price |
| `order_items` | item_id, order_id, product_id, quantity, discount |